In [0]:
%sql
-- 1. Crear el catálogo del equipo
CREATE CATALOG IF NOT EXISTS stasys_team2;

-- 2. Seleccionar el catálogo
USE CATALOG stasys_team2;

-- ----------------------------------------------------
-- ESQUEMA DE TEAM 02: 
-- ----------------------------------------------------

-- 3. Crear tu esquema de trabajo
CREATE SCHEMA IF NOT EXISTS gianmarco;

-- 4. Verificar los esquemas disponibles
SHOW SCHEMAS IN stasys_team2;

-- 5. Comprobar permisos y ubicación
SELECT current_catalog(), current_schema();

In [0]:
%sql
CREATE TABLE IF NOT EXISTS
    stasys_team2.gianmarco.consultas_denuncias_silver
USING CSV
OPTIONS (
    header = "true",
    inferSchema = "true"
)
LOCATION 's3://data-lagoon-y/silver/cleaned/consultas_denuncias_2026_v1/2026/10/13/20261013_150857_891495/CONSULTAS_DENUNCIAS_2026_v1_silver.csv';

In [0]:
%sql
-- Confirmar que la tabla existe.
DESCRIBE TABLE
    stasys_team2.gianmarco.consultas_denuncias_silver;

-- Consultar una muestra.
SELECT *
FROM stasys_team2.gianmarco.consultas_denuncias_silver
LIMIT 20;

-- Contar los registros.
SELECT COUNT(*) AS total_registros
FROM stasys_team2.gianmarco.consultas_denuncias_silver;


In [0]:
%sql
-- Consulta 1 — Cantidad de solicitudes por tipo

SELECT
    DE_TIPO_SOLICITUD AS tipo_solicitud,
    COUNT(*) AS total_solicitudes
FROM stasys_team2.gianmarco.consultas_denuncias_silver
GROUP BY DE_TIPO_SOLICITUD
ORDER BY total_solicitudes DESC;

In [0]:
%sql
-- Consulta 2 — Evolución mensual de las solicitudes

SELECT
    DATE_TRUNC('MONTH', FE_REGISTRO_FECHA) AS mes,
    COUNT(*) AS cantidad_solicitudes
FROM stasys_team2.gianmarco.consultas_denuncias_silver
WHERE FE_REGISTRO_FECHA IS NOT NULL
GROUP BY DATE_TRUNC('MONTH', FE_REGISTRO_FECHA)
ORDER BY cantidad_solicitudes DESC;

In [0]:
%sql
-- Consulta 3 — Ranking de las diez instituciones con más solicitudes

SELECT
    DE_INSTITUCION,
    DE_TIPO_INST,
    COUNT(*) AS cantidad_solicitudes
FROM stasys_team2.gianmarco.consultas_denuncias_silver
WHERE DE_INSTITUCION IS NOT NULL
GROUP BY
    DE_INSTITUCION,
    DE_TIPO_INST
ORDER BY cantidad_solicitudes DESC
LIMIT 10;


In [0]:
%sql
-- Consulta 4 — Comparación de solicitudes según la naturaleza de la institución


SELECT
    DE_NATURALEZA_INST,
    COUNT(*) AS total_solicitudes,
    COUNT(DISTINCT DE_INSTITUCION)
        AS instituciones_distintas,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS porcentaje_solicitudes
FROM stasys_team2.gianmarco.consultas_denuncias_silver
WHERE DE_NATURALEZA_INST IS NOT NULL
GROUP BY DE_NATURALEZA_INST
ORDER BY total_solicitudes DESC;


In [0]:
-- Consulta 5 — Distribución de solicitudes por grupos de edad y género


from pyspark.sql import functions as F

# Leer la tabla SILVER
df = spark.table(
    "stasys_team2.gianmarco.consultas_denuncias_silver"
)

# 1. Convertir y validar edades
df_validado = df.withColumn(
    "edad_numerica",
    F.expr("try_cast(DE_SOLICITANTE_EDAD AS INT)")
).withColumn(
    "edad_valida",
    F.when(
        F.col("edad_numerica").between(0, 120),
        F.col("edad_numerica")
    )
)

# 2. Crear grupos de edad
df_grupos = df_validado.withColumn(
    "grupo_edad",
    F.when(F.col("edad_valida").isNull(), "Sin Edad")
    .when(F.col("edad_valida") < 18, "0-17")
    .when(F.col("edad_valida") < 30, "18-29")
    .when(F.col("edad_valida") < 40, "30-39")
    .when(F.col("edad_valida") < 50, "40-49")
    .when(F.col("edad_valida") < 60, "50-59")
    .when(F.col("edad_valida") < 70, "60-69")
    .otherwise("70+")
).withColumn(
    "genero",
    F.coalesce(
        F.col("DE_SOLICITANTE_GENERO"),
        F.lit("No especificado")
    )
)

# 3. Reporte de validación
df_validado.select(
    F.count("*").alias("total_registros"),
    F.count("edad_numerica").alias("edades_numericas"),
    F.count("edad_valida").alias("edades_validas"),
    (
        F.count("*") - F.count("edad_valida")
    ).alias("edades_invalidas_o_nulas")
).show(truncate=False)

# 4. Distribución por edad y género
distribucion = df_grupos.groupBy(
    "grupo_edad",
    "genero"
).agg(
    F.count("*").alias("cantidad_solicitudes"),
    F.round(F.avg("edad_valida"), 2).alias("edad_promedio"),
    F.min("edad_valida").alias("edad_minima"),
    F.max("edad_valida").alias("edad_maxima")
)

# 5. Ordenar y mostrar resultados
orden = [
    "0-17", "18-29", "30-39", "40-49",
    "50-59", "60-69", "70+", "Sin Edad"
]

distribucion = distribucion.withColumn(
    "orden_edad",
    F.when(F.col("grupo_edad") == "0-17", 1)
    .when(F.col("grupo_edad") == "18-29", 2)
    .when(F.col("grupo_edad") == "30-39", 3)
    .when(F.col("grupo_edad") == "40-49", 4)
    .when(F.col("grupo_edad") == "50-59", 5)
    .when(F.col("grupo_edad") == "60-69", 6)
    .when(F.col("grupo_edad") == "70+", 7)
    .otherwise(8)
).orderBy(
    "orden_edad",
    "genero"
).drop("orden_edad")

distribucion.show(100, truncate=False)
GROUP BY
    CASE
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) < 18
            THEN 'Menor de 18'
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) BETWEEN 18 AND 29
            THEN '18-29'
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) BETWEEN 30 AND 44
            THEN '30-44'
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) BETWEEN 45 AND 59
            THEN '45-59'
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) >= 60
            THEN '60 o más'
        ELSE 'Edad no disponible'
    END,
    DE_SOLICITANTE_GENERO
ORDER BY grupo_edad, genero;